# 海外债 重构

本文档用于海外债数据的获取、处理和分析。

## 1. 项目概述

### 1.1 功能说明
- 海外债数据采集与处理
- 数据库存储与更新
- 数据分析与可视化

### 1.2 数据来源
- Wind金融终端
- 同花顺iFinD
- 企业预警通

### 1.3 输出
- 海外债数据表
- 分析报告

## 2. 环境配置

### 2.1 导入依赖库

In [ ]:
# 标准库
import os
import sys
import json
import datetime
from datetime import datetime, timedelta
from time import sleep
import time
import random
import warnings

warnings.filterwarnings('ignore')

# 数据处理
import pandas as pd
import numpy as np
import chardet

# 数据库
import pymysql
import sqlalchemy
from sqlalchemy import inspect, MetaData, Table, Column, Text, text
from sqlalchemy import create_engine
import sqlalchemy.pool

# 数据源接口（根据实际情况启用）
# from WindPy import w
# from iFinDPy import *

# 可视化
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

# 配置
from config import Config

print("环境配置完成")

### 2.2 初始化配置

In [ ]:
# 加载配置
config = Config()

# 数据库连接
db_config = config.get_database_config()
sql_engine = create_engine(
    'mysql+pymysql://{user}:{password}@{host}:{port}/{database}'.format(
        user=db_config['user'],
        password=db_config['password'],
        host=db_config['host'],
        port=db_config['port'],
        database=db_config['database']
    ),
    poolclass=sqlalchemy.pool.NullPool,
    pool_recycle=3600
)

print(f"数据库连接配置: {db_config['host']}/{db_config['database']}")

### 2.3 初始化数据源接口

In [ ]:
# 初始化Wind接口（如需要）
# w.start()
# print("Wind接口初始化完成")

# 初始化iFinD接口（如需要）
# ifind_user = os.environ.get('IFIND_USER', '')
# ifind_password = os.environ.get('IFIND_PASSWORD', '')
# THS_iFinDLogin(ifind_user, ifind_password)
# print("iFinD接口初始化完成")

print("数据源接口初始化（请根据需要启用）")

## 3. 数据获取

### 3.1 海外债数据获取函数

In [ ]:
def fetch_overseas_bond_data(start_date, end_date, source='ifind'):
    """
    获取海外债数据
    
    参数:
        start_date: 开始日期 (YYYYMMDD 或 YYYY-MM-DD)
        end_date: 结束日期 (YYYYMMDD 或 YYYY-MM-DD)
        source: 数据源 ('wind' 或 'ifind')
    
    返回:
        DataFrame: 海外债数据
    """
    # 转换日期格式
    if isinstance(start_date, str) and '-' in start_date:
        start_date = start_date.replace('-', '')
    if isinstance(end_date, str) and '-' in end_date:
        end_date = end_date.replace('-', '')
    
    df = None
    
    if source == 'wind':
        # Wind数据获取示例
        # error_code, df = w.wset(
        #     "overseasbondissue",
        #     f"startdate={start_date};enddate={end_date}",
        #     usedf=True
        # )
        print("Wind数据获取接口待配置")
        
    elif source == 'ifind':
        # iFinD数据获取示例
        # df = THS_DR('p04524', f'sdate={start_date};edate={end_date};zqlx=...', 'fields', 'format:dataframe')
        # df = df.data
        print("iFinD数据获取接口待配置")
    
    return df


def get_date_range(days_back=30, days_forward=30):
    """
    获取日期范围
    
    参数:
        days_back: 向前推算天数
        days_forward: 向后推算天数
    
    返回:
        tuple: (start_date, end_date) 格式为YYYYMMDD
    """
    current_date = datetime.now()
    start_date = (current_date + timedelta(days=days_back)).strftime('%Y%m%d')
    end_date = (current_date + timedelta(days=days_forward)).strftime('%Y%m%d')
    return start_date, end_date


print("数据获取函数定义完成")

### 3.2 执行数据获取

In [ ]:
# 获取日期范围
start_date, end_date = get_date_range(days_back=1, days_forward=30)
print(f"数据获取范围: {start_date} 至 {end_date}")

# 获取海外债数据
# bond_data = fetch_overseas_bond_data(start_date, end_date, source='ifind')

# if bond_data is not None and not bond_data.empty:
#     print(f"获取到 {len(bond_data)} 条记录")
#     print(bond_data.head())
# else:
#     print("未获取到数据或数据为空")

print("请根据实际数据源配置数据获取逻辑")

## 4. 数据处理

### 4.1 数据清洗函数

In [ ]:
def clean_bond_data(df):
    """
    清洗债券数据
    
    参数:
        df: 原始数据DataFrame
    
    返回:
        DataFrame: 清洗后的数据
    """
    if df is None or df.empty:
        return df
    
    # 复制数据避免修改原始数据
    df = df.copy()
    
    # 处理日期列
    if 'dt' in df.columns:
        df['dt'] = pd.to_datetime(df['dt'], errors='coerce')
        df = df.dropna(subset=['dt'])
        df['dt'] = df['dt'].dt.strftime('%Y-%m-%d')
    
    # 替换NaN值为None
    df = df.replace({pd.NA: None, pd.NaT: None, float('nan'): None})
    
    # 重置索引
    df = df.reset_index(drop=True)
    
    return df


def validate_bond_data(df):
    """
    验证债券数据完整性
    
    参数:
        df: 数据DataFrame
    
    返回:
        bool: 数据是否有效
    """
    if df is None or df.empty:
        print("数据为空")
        return False
    
    # 检查必要列
    required_columns = ['dt']  # 根据实际需求调整
    missing_cols = [col for col in required_columns if col not in df.columns]
    if missing_cols:
        print(f"缺少必要列: {missing_cols}")
        return False
    
    return True


print("数据处理函数定义完成")

### 4.2 执行数据清洗

In [ ]:
# 清洗数据
# cleaned_data = clean_bond_data(bond_data)

# if validate_bond_data(cleaned_data):
#     print(f"数据清洗完成，有效记录数: {len(cleaned_data)}")
#     print(cleaned_data.info())
# else:
#     print("数据验证失败")

print("数据清洗步骤待执行")

## 5. 核心逻辑

### 5.1 数据库操作函数

In [ ]:
def generate_column_mapping(columns):
    """
    生成列名映射（中文转英文用于数据库操作）
    
    参数:
        columns: 列名列表
    
    返回:
        dict: 列名映射字典
    """
    return {col: f"col_{i}" for i, col in enumerate(columns)}


def change_column_names(table_name, column_mapping, engine, to_english=True):
    """
    修改表列名
    
    参数:
        table_name: 表名
        column_mapping: 列名映射字典
        engine: SQLAlchemy引擎
        to_english: 是否转换为英文列名
    """
    with engine.begin() as connection:
        for original_name, new_name in column_mapping.items():
            if to_english:
                connection.execute(text(
                    f"ALTER TABLE `{table_name}` CHANGE `{original_name}` `{new_name}` VARCHAR(255);"
                ))
            else:
                connection.execute(text(
                    f"ALTER TABLE `{table_name}` CHANGE `{new_name}` `{original_name}` VARCHAR(255);"
                ))


def upsert_data_to_db(df, table_name, engine):
    """
    将数据更新插入到数据库
    
    参数:
        df: 数据DataFrame
        table_name: 目标表名
        engine: SQLAlchemy引擎
    """
    if df is None or df.empty:
        print("无有效数据需要插入")
        return
    
    # 生成列名映射
    column_mapping = generate_column_mapping(df.columns)
    df_renamed = df.rename(columns=column_mapping)
    
    inspector = inspect(engine)
    table_exists = inspector.has_table(table_name)
    
    if table_exists:
        # 获取现有表的列名
        existing_columns = inspector.get_columns(table_name)
        existing_columns_names = [col['name'] for col in existing_columns]
        df_columns = df_renamed.columns.tolist()
        
        # 检查并添加缺失的列
        for col in df_columns:
            if col not in existing_columns_names:
                col_type = "FLOAT" if pd.api.types.is_numeric_dtype(df_renamed[col]) else "VARCHAR(255)"
                with engine.begin() as connection:
                    connection.execute(text(f"ALTER TABLE `{table_name}` ADD COLUMN `{col}` {col_type};"))
        
        # 构建插入更新语句
        insert_columns = ', '.join([f"`{col}`" for col in df_columns])
        update_columns = ', '.join([f"`{col}` = VALUES(`{col}`)" for col in df_columns])
        value_placeholders = ', '.join([f":{col}" for col in df_columns])
        
        insert_query = text(f"""
        INSERT INTO `{table_name}` ({insert_columns})
        VALUES ({value_placeholders})
        ON DUPLICATE KEY UPDATE {update_columns};
        """)
        
        # 执行插入
        with engine.begin() as connection:
            for _, row in df_renamed.iterrows():
                try:
                    params = {col: row[col] for col in df_columns}
                    connection.execute(insert_query, params)
                except Exception as e:
                    print(f"插入数据失败: {row.to_dict()}")
                    print(f"错误: {e}")
        
        print(f"数据更新完成，表: {table_name}")
    else:
        # 表不存在，直接创建
        df_renamed.to_sql(table_name, engine, if_exists='replace', index=False)
        print(f"创建新表: {table_name}")


def remove_duplicates(table_name, engine, unique_column='sec_name', code_column='trade_code'):
    """
    删除重复数据
    
    参数:
        table_name: 表名
        engine: SQLAlchemy引擎
        unique_column: 唯一标识列
        code_column: 代码列
    """
    with engine.begin() as connection:
        connection.execute(text(f"""
        CREATE TEMPORARY TABLE temp_table AS
        SELECT * FROM `{table_name}`
        WHERE {unique_column} IN (
            SELECT {unique_column}
            FROM `{table_name}`
            GROUP BY {unique_column}
            HAVING COUNT(*) = 1
        )
        UNION
        SELECT * FROM `{table_name}`
        WHERE {code_column} IN (
            SELECT MIN({code_column})
            FROM `{table_name}`
            WHERE {code_column} NOT LIKE 'S%%'
            GROUP BY {unique_column}
            HAVING COUNT(*) > 1
        );
        
        DELETE FROM `{table_name}`;
        
        INSERT INTO `{table_name}` SELECT * FROM temp_table;
        
        DROP TEMPORARY TABLE temp_table;
        """))
    print(f"重复数据清理完成")


print("数据库操作函数定义完成")

### 5.2 执行数据库操作

In [ ]:
# 表名
TABLE_NAME = '海外债数据'

# 生成列名映射
# column_mapping = generate_column_mapping(cleaned_data.columns)

# 修改表的列名为英文（便于数据库操作）
# change_column_names(TABLE_NAME, column_mapping, sql_engine, to_english=True)

# 插入数据
# upsert_data_to_db(cleaned_data, TABLE_NAME, sql_engine)

# 修改表的列名回中文
# change_column_names(TABLE_NAME, column_mapping, sql_engine, to_english=False)

# 删除重复数据
# remove_duplicates(TABLE_NAME, sql_engine)

print("数据库操作步骤待执行")

## 6. 执行与测试

### 6.1 完整执行流程

In [ ]:
def main():
    """
    主执行函数
    """
    print("=" * 50)
    print("海外债数据处理流程开始")
    print("=" * 50)
    
    # 1. 获取日期范围
    start_date, end_date = get_date_range(days_back=1, days_forward=30)
    print(f"\n[步骤1] 日期范围: {start_date} 至 {end_date}")
    
    # 2. 获取数据
    print("\n[步骤2] 获取海外债数据...")
    # bond_data = fetch_overseas_bond_data(start_date, end_date)
    # if bond_data is None or bond_data.empty:
    #     print("未获取到数据，流程结束")
    #     return
    # print(f"获取到 {len(bond_data)} 条记录")
    
    # 3. 清洗数据
    print("\n[步骤3] 清洗数据...")
    # cleaned_data = clean_bond_data(bond_data)
    # if not validate_bond_data(cleaned_data):
    #     print("数据验证失败，流程结束")
    #     return
    
    # 4. 存储数据
    print("\n[步骤4] 存储数据到数据库...")
    # upsert_data_to_db(cleaned_data, TABLE_NAME, sql_engine)
    
    # 5. 清理重复数据
    print("\n[步骤5] 清理重复数据...")
    # remove_duplicates(TABLE_NAME, sql_engine)
    
    print("\n" + "=" * 50)
    print("海外债数据处理流程完成")
    print("=" * 50)

# 执行主函数
# main()

print("主执行函数定义完成，请取消注释以执行")

### 6.2 数据查询验证

In [ ]:
# 查询数据库中的数据
def query_data(table_name, engine, limit=10):
    """
    查询数据库中的数据
    
    参数:
        table_name: 表名
        engine: SQLAlchemy引擎
        limit: 返回记录数限制
    
    返回:
        DataFrame: 查询结果
    """
    query = f"SELECT * FROM `{table_name}` ORDER BY dt DESC LIMIT {limit}"
    with engine.begin() as connection:
        result = pd.read_sql(query, con=connection)
    return result

# 验证数据
# result = query_data(TABLE_NAME, sql_engine)
# print(result)

print("查询验证函数定义完成")

## 7. 工具函数（可复用）

### 7.1 日期工具

In [ ]:
def get_current_datetime():
    """
    获取当前日期时间
    
    返回:
        str: YYYY-MM-DD HH:MM:SS 格式的日期时间
    """
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S')


def date_format_convert(date_str, from_format='%Y-%m-%d', to_format='%Y%m%d'):
    """
    日期格式转换
    
    参数:
        date_str: 日期字符串
        from_format: 原格式
        to_format: 目标格式
    
    返回:
        str: 转换后的日期字符串
    """
    try:
        dt = datetime.strptime(date_str, from_format)
        return dt.strftime(to_format)
    except ValueError:
        return date_str


def get_trading_days(start_date, end_date):
    """
    获取交易日列表（需要Wind或iFinD接口）
    
    参数:
        start_date: 开始日期
        end_date: 结束日期
    
    返回:
        list: 交易日列表
    """
    # 使用Wind接口获取交易日
    # trading_days = w.tdays(start_date, end_date)
    # return [d.strftime('%Y-%m-%d') for d in trading_days.Data[0]]
    
    # 临时返回工作日列表
    date_range = pd.date_range(start=start_date, end=end_date, freq='B')
    return [d.strftime('%Y-%m-%d') for d in date_range]


print("日期工具函数定义完成")

### 7.2 数据库工具

In [ ]:
def get_table_columns(table_name, engine):
    """
    获取表的列名列表
    
    参数:
        table_name: 表名
        engine: SQLAlchemy引擎
    
    返回:
        list: 列名列表
    """
    inspector = inspect(engine)
    columns = inspector.get_columns(table_name)
    return [col['name'] for col in columns]


def get_table_row_count(table_name, engine):
    """
    获取表的行数
    
    参数:
        table_name: 表名
        engine: SQLAlchemy引擎
    
    返回:
        int: 行数
    """
    query = f"SELECT COUNT(*) as cnt FROM `{table_name}`"
    with engine.begin() as connection:
        result = pd.read_sql(query, con=connection)
    return result['cnt'].iloc[0]


def execute_sql(sql, engine):
    """
    执行SQL语句
    
    参数:
        sql: SQL语句
        engine: SQLAlchemy引擎
    
    返回:
        DataFrame: 查询结果（如果是查询语句）
    """
    with engine.begin() as connection:
        if sql.strip().upper().startswith('SELECT'):
            return pd.read_sql(sql, con=connection)
        else:
            connection.execute(text(sql))
            return None


print("数据库工具函数定义完成")

### 7.3 文件工具

In [ ]:
def read_excel_template(file_path, sheet_name=None):
    """
    读取Excel模板文件
    
    参数:
        file_path: 文件路径
        sheet_name: 工作表名（可选）
    
    返回:
        DataFrame: 数据
    """
    if sheet_name:
        return pd.read_excel(file_path, sheet_name=sheet_name)
    return pd.read_excel(file_path)


def export_to_excel(df, file_path, sheet_name='Sheet1'):
    """
    导出数据到Excel文件
    
    参数:
        df: 数据DataFrame
        file_path: 输出文件路径
        sheet_name: 工作表名
    """
    df.to_excel(file_path, sheet_name=sheet_name, index=False)
    print(f"数据已导出到: {file_path}")


print("文件工具函数定义完成")

---
**重构完成**

本文档提供了海外债数据处理的基础框架，请根据实际需求配置数据源和业务逻辑。